In [ ]:
#| default_exp crypto

# Crypto

> AES-256-CBC encryption/decryption and signature utilities for WeCom.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import base64, hashlib, os, struct
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from solveit_wxbot.config import AES_KEY, CORP_ID

## PKCS#7 填充

WeCom 使用 AES-256-CBC 加密，要求明文长度为块大小（32字节）的整数倍。`pkcs7_pad` 在末尾追加填充字节，`pkcs7_unpad` 则将其还原。

In [ ]:
#| export
def _pkcs7_pad(
    data: bytes,  # Data to pad
    bs: int = 32  # Block size; WeCom requires 32
) -> bytes:
    "Pad `data` to a multiple of `bs` bytes using PKCS#7."
    n = bs - len(data) % bs
    return data + bytes([n]) * n

def _pkcs7_unpad(
    data: bytes  # PKCS#7-padded bytes
) -> bytes:
    "Remove PKCS#7 padding from `data`."
    return data[: -data[-1]]

## 消息签名

WeCom 通过对若干字段排序后拼接再 SHA1 哈希来验证请求来源。`msg_sig` 封装了这一逻辑，供 URL 验证和消息接收两处复用。

In [ ]:
#| export
def msg_sig(
    *parts: str  # Strings to sort, concatenate, and SHA1-hash
) -> str:
    "WeCom message signature: sort `parts`, join, and return SHA1 hex digest."
    return hashlib.sha1(''.join(sorted(parts)).encode()).hexdigest()

## 加解密

`decrypt` 将 WeCom 下发的 Base64 密文还原为 XML 明文和企业 ID；`encrypt` 则将明文 XML 加密为 Base64，用于被动回复。两者均使用 AES-256-CBC，IV 取自 `AES_KEY` 前16字节。

In [ ]:
#| export
def decrypt(
    encrypted: str  # Base64-encoded AES-256-CBC ciphertext from WeCom
) -> tuple:
    "Decrypt a WeCom message; returns `(xml_text, corp_id)`."
    dec = Cipher(algorithms.AES(AES_KEY), modes.CBC(AES_KEY[:16])).decryptor()
    raw = pkcs7_unpad(dec.update(base64.b64decode(encrypted)) + dec.finalize())
    n = struct.unpack('>I', raw[16:20])[0]
    return raw[20:20+n].decode(), raw[20+n:].decode()

def encrypt(
    xml: str  # Plaintext XML to encrypt for a WeCom passive reply
) -> str:
    "Encrypt `xml` using AES-256-CBC for WeCom; returns a Base64 string."
    msg = xml.encode()
    body = pkcs7_pad(os.urandom(16) + struct.pack('>I', len(msg)) + msg + CORP_ID.encode())
    enc = Cipher(algorithms.AES(AES_KEY), modes.CBC(AES_KEY[:16])).encryptor()
    return base64.b64encode(enc.update(body) + enc.finalize()).decode()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()